# tools

> This module provides tools for AI agents to explore and interact with linked data structures. These tools are designed to work with the Claudette library to enable Claude to navigate and understand JSON-LD data.


## Overview

The tools in this module enable:

1. **Vocabulary exploration**: Finding and understanding terms in vocabularies
2. **Relationship navigation**: Following connections between concepts
3. **Dataset exploration**: Examining the structure of datasets
4. **Search**: Finding specific information in datasets
5. **Evidence collection**: Gathering evidence about specific topics

All tools return markdown-formatted strings that Claude can interpret and use to build comprehensive responses.

## Vocabulary Exploration

```python
@tool
def find_vocabulary_term(
    term: str,  # Term to find (e.g., "Person", "schema:Person", or full URI)
    vocab_name: str = "default",  # Name of vocabulary to search in
    search_type: str = "all"  # Search by: "id", "label", "type", or "all"
) -> str:  # Definition and details of the term
```

This tool allows Claude to find and understand terms in vocabularies. It can search by ID, label, or type, and returns a detailed description of the term including its definition and relationships.

Example usage:
```python
find_vocabulary_term("Person", search_type="id")
```

## Relationship Navigation

```python
@tool
def follow_relationship(
    entity_id: str,  # ID of the entity to start from
    relationship: str = None,  # Relationship to follow (or None to list all)
    include_inverse: bool = False  # Whether to include inverse relationships
) -> str:  # Related entities or available relationships
```

This tool enables Claude to navigate the connections between concepts in a vocabulary. It can list all available relationships for an entity or follow a specific relationship to find related entities.

Example usage:
```python
# List all relationships for Person
follow_relationship("Person")

# Follow a specific relationship
follow_relationship("Person", "rdfs:subClassOf")

# Include inverse relationships
follow_relationship("Person", include_inverse=True)
```

## Dataset Exploration

```python
@tool
def explore_dataset(
    path: str = "",  # Path to explore in the dataset
    max_size: int = 4000  # Maximum size of returned JSON
) -> str:  # Dataset fragment
```

This tool allows Claude to explore the structure of a dataset at a specific path. It returns a human-readable representation of the data at that path, including key properties and their values.

Example usage:
```python
# Explore the root level
explore_dataset()

# Explore a specific path
explore_dataset("recordSet[0].field")
```

## Search

```python
@tool
def search_dataset(
    query: str,  # Text to search for in the dataset
    case_sensitive: bool = False  # Whether search should be case sensitive
) -> str:  # Search results
```

This tool enables Claude to search for specific text or patterns across the dataset. It returns a list of paths where the query was found, grouped by keys and values.

Example usage:
```python
# Search for a term
search_dataset("machine learning")

# Case-sensitive search
search_dataset("RecordSet", case_sensitive=True)
```

## Evidence Collection

```python
@tool
def get_dataset_evidence(
    topic: str,  # Topic to find evidence about
    max_results: int = 5  # Maximum number of evidence items to return
) -> str:  # Evidence from the dataset
```

This tool helps Claude gather structured evidence about a specific topic from the dataset. It returns a collection of evidence items, each with context information and the actual value found.

Example usage:
```python
# Get evidence about a topic
get_dataset_evidence("machine learning")

# Limit the number of results
get_dataset_evidence("feature", max_results=1)
```

## Usage with Claude

These tools are designed to be used with Claude via Claudette's `tool` decorator. To use them in an agent, you need to:

1. Load your vocabularies and datasets into global variables named `kb` and `dataset_kb`
2. Create a Chat instance with these tools
3. Run the agent with a prompt that instructs Claude how to use the tools

Example:
```python
from claudette import Chat

# Load vocabulary and dataset
kb = LinkedDataKnowledge()
kb.fetch_vocabulary("https://schema.org/")

dataset_kb = LinkedDataKnowledge(my_dataset)

# Create an agent with the tools
agent = Chat(model="claude-3-5-sonnet-20241022", tools=[
    find_vocabulary_term,
    follow_relationship,
    explore_dataset,
    search_dataset,
    get_dataset_evidence
])

# Run the agent
result = agent.toolloop("""
Please explore this dataset and tell me what it contains.
Start by examining the overall structure, then look for information about
machine learning features.
""")
```

By combining these tools, Claude can navigate and understand complex linked data structures, following connections between concepts and gathering evidence to provide comprehensive responses.


In [ ]:
#| default_exp tools

In [ ]:
#| exports

from fastcore.basics import *
from fastcore.meta import *
from fastcore.test import *
from IPython.display import Markdown, display
from cogitarelink.core import LinkedDataKnowledge
from cogitarelink.navigation import *
from claudette import tool, Chat, models
import json
from typing import List, Dict, Any, Optional, Union, Set


In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
models

['claude-3-opus-20240229',
 'claude-3-7-sonnet-20250219',
 'claude-3-5-sonnet-20241022',
 'claude-3-haiku-20240307',
 'claude-3-5-haiku-20241022']

In [ ]:
model=models[1]
model

'claude-3-7-sonnet-20250219'

In [ ]:
#| export
@tool
def find_vocabulary_term(
    term: str,  # Term to find (e.g., "Person", "schema:Person", or full URI)
    vocab_name: str = "default",  # Name of vocabulary to search in
    search_type: str = "all"  # Search by: "id", "label", "type", or "all"
) -> str:  # Definition and details of the term
    """Find a specific term in a vocabulary and return its definition.
    
    Args:
        term: The term to search for (can be a partial name, prefixed term, or full URI)
        vocab_name: Name of the vocabulary to search in (use "default" for the main vocabulary)
        search_type: How to search ("id", "label", "type", or "all")
        
    Returns:
        A markdown-formatted description of the term(s) found
    """
    # Get the knowledge base from the global context
    if 'kb' not in globals():
        return "No vocabulary loaded. Please load a vocabulary first."
    
    kb = globals()['kb']
    
    # Find entities matching the term
    entities = []
    if search_type in ["id", "all"]:
        entities.extend(kb.find_entity(entity_id=term))
    if search_type in ["label", "all"]:
        entities.extend(kb.find_entity(label=term))
    if search_type in ["type", "all"]:
        # For type searches, we need to check the @type property directly
        for entity in kb.data.get('@graph', []):
            entity_type = entity.get('@type', '')
            # Handle both list and string types
            if isinstance(entity_type, list):
                if any(term in t for t in entity_type):
                    entities.append(entity)
            elif term in str(entity_type):
                entities.append(entity)
    
    # Remove duplicates by ID
    unique_entities = []
    seen_ids = set()
    for entity in entities:
        entity_id = entity.get('@id', '')
        if entity_id not in seen_ids:
            seen_ids.add(entity_id)
            unique_entities.append(entity)
    
    if not unique_entities:
        return f"No terms found matching '{term}' in the vocabulary."
    
    # Format the results
    output = [f"# Found {len(unique_entities)} terms matching '{term}'"]
    
    for i, entity in enumerate(unique_entities):
        output.append(f"\n## Term {i+1}: {entity.get('@id', 'Unknown ID').split('/')[-1]}")
        output.append(kb.get_entity_description(entity))
    
    return "\n".join(output)


In [ ]:
#| eval: False
def test_find_vocabulary_term():
    # Setup a test knowledge base
    global kb
    kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person",
                "rdfs:comment": "A person (alive, dead, undead, or fictional)."
            },
            {
                "@id": "http://example.org/name",
                "@type": "rdf:Property",
                "rdfs:label": "name",
                "rdfs:domain": {"@id": "http://example.org/Person"}
            }
        ]
    })
    
    # Test 1: Find by ID
    result = find_vocabulary_term("Person", search_type="id")
    assert("Found 1 terms" in result)
    assert("Person" in result)
    assert("A person (alive, dead, undead, or fictional)" in result)
    
    # Test 2: Find by label
    result = find_vocabulary_term("name", search_type="label")
    assert("Found 1 terms" in result)
    assert("name" in result)
    
    # Test 3: Find by type
    result = find_vocabulary_term("rdfs:Class", search_type="type")
    assert("Found 1 terms" in result)
    assert("Person" in result)
    
    # Test 4: No results
    result = find_vocabulary_term("NonExistent")
    assert("No terms found" in result)
    
    # Test 5: No vocabulary loaded
    # Save a reference to kb before deleting it
    kb_backup = kb
    del globals()['kb']
    result = find_vocabulary_term("Person")
    assert("No vocabulary loaded" in result)
    
    # Restore the knowledge base for other tests
    globals()['kb'] = kb_backup


In [ ]:
test_find_vocabulary_term()

In [ ]:
#| export
@tool
def follow_relationship(
    entity_id: str,  # ID of the entity to start from
    relationship: str = None,  # Relationship to follow (or None to list all)
    include_inverse: bool = False  # Whether to include inverse relationships
) -> str:  # Related entities or available relationships
    """Follow a relationship from an entity to find related entities.
    
    Args:
        entity_id: The ID of the entity to start from
        relationship: The relationship to follow, or None to list available relationships
        include_inverse: Whether to include relationships where this entity is the object
        
    Returns:
        A markdown-formatted description of the related entities or available relationships
    """
    # Get the knowledge base from the global context
    if 'kb' not in globals():
        return "No vocabulary loaded. Please load a vocabulary first."
    
    kb = globals()['kb']
    
    # Find the starting entity
    entities = kb.find_entity(entity_id=entity_id)
    if not entities:
        return f"Entity '{entity_id}' not found."
    
    entity = entities[0]
    entity_id_full = entity.get('@id', 'Unknown')
    entity_label = entity_id_full.split('/')[-1]
    
    # If no relationship specified, list available relationships
    if relationship is None:
        relationships = kb.follow_relationship(entity_id_full, None, include_inverse)
        
        if not relationships:
            return f"Entity '{entity_label}' has no relationships."
        
        output = [f"# Available relationships for '{entity_label}'"]
        
        # Group relationships by type (direct vs inverse)
        direct_rels = [r for r in relationships if not r.startswith('^')]
        inverse_rels = [r[1:] for r in relationships if r.startswith('^')]
        
        if direct_rels:
            output.append("\n## Direct relationships (where this entity is the subject)")
            for rel in direct_rels:
                rel_label = rel.split('/')[-1] if '/' in rel else rel
                output.append(f"- {rel_label}")
        
        if inverse_rels:
            output.append("\n## Inverse relationships (where this entity is the object)")
            for rel in inverse_rels:
                rel_label = rel.split('/')[-1] if '/' in rel else rel
                output.append(f"- {rel_label}")
        
        return "\n".join(output)
    
    # Follow the specified relationship
    related_entities = kb.follow_relationship(entity_id_full, relationship)
    
    if not related_entities:
        return f"No entities found related to '{entity_label}' via '{relationship}'."
    
    # Format the results
    output = [f"# Entities related to '{entity_label}' via '{relationship}'"]
    
    for i, related in enumerate(related_entities):
        related_id = related.get('@id', 'Unknown')
        related_label = related_id.split('/')[-1]
        output.append(f"\n## {i+1}. {related_label}")
        output.append(kb.get_entity_description(related))
    
    return "\n".join(output)

In [ ]:
#| test
def test_follow_relationship():
    # Setup a test knowledge base
    global kb
    kb = LinkedDataKnowledge({
        "@context": {},
        "@graph": [
            {
                "@id": "http://example.org/Person",
                "@type": "rdfs:Class",
                "rdfs:label": "Person",
                "rdfs:subClassOf": {"@id": "http://example.org/Thing"}
            },
            {
                "@id": "http://example.org/Thing",
                "@type": "rdfs:Class",
                "rdfs:label": "Thing"
            },
            {
                "@id": "http://example.org/name",
                "@type": "rdf:Property",
                "rdfs:domain": {"@id": "http://example.org/Person"}
            }
        ]
    })
    
    # Test 1: List available relationships
    result = follow_relationship("Person")
    assert("Available relationships" in result)
    assert("rdfs:label" in result)
    assert("rdfs:subClassOf" in result)
    
    # Test 2: Follow a specific relationship
    result = follow_relationship("Person", "rdfs:subClassOf")
    assert("Entities related to 'Person'" in result)
    assert("Thing" in result)
    
    # Test 3: Include inverse relationships
    result = follow_relationship("Person", include_inverse=True)
    assert("Inverse relationships" in result)
    assert("rdfs:domain" in result)
    
    # Test 4: Follow an inverse relationship
    result = follow_relationship("Person", "^rdfs:domain")
    assert("Entities related to 'Person'" in result)
    assert("name" in result)
    
    # Test 5: Non-existent relationship
    result = follow_relationship("Person", "nonexistent")
    assert("No entities found" in result)
    
    # Test 6: Non-existent entity
    result = follow_relationship("NonExistent")
    assert("not found" in result)

In [ ]:
test_follow_relationship()

In [ ]:
#| export
@tool
def explore_dataset(
    path: str = "",  # Path to explore in the dataset
    max_size: int = 4000  # Maximum size of returned JSON
) -> str:  # Dataset fragment
    """Explore the dataset structure at the given path.
    
    Args:
        path: The path to explore, using dot notation (e.g., "recordSet[0].field")
        max_size: Maximum size of the returned JSON in characters
        
    Returns:
        A markdown-formatted representation of the data at the specified path
    """
    # Get the dataset knowledge base from the global context
    if 'dataset_kb' not in globals():
        return "No dataset loaded. Please load a dataset first."
    
    dataset_kb = globals()['dataset_kb']
    
    try:
        # Navigate to the specified path
        current = dataset_kb.data
        if path:
            parts = path.split(".")
            for part in parts:
                if "[" in part and part.endswith("]"):
                    name, idx_str = part.split("[", 1)
                    idx = int(idx_str[:-1])
                    if name:
                        current = current[name][idx]
                    else:
                        current = current[idx]
                else:
                    current = current[part]
        
        # Format the result based on its type
        if isinstance(current, dict):
            # For dictionaries, show a structured view
            output = [f"# Structure at path: {path or 'root'}"]
            
            # Special handling for entities with @id and @type
            if '@id' in current:
                output.append(f"**ID**: `{current['@id']}`")
            if '@type' in current:
                type_val = current['@type']
                if isinstance(type_val, list):
                    output.append(f"**Type**: {', '.join([f'`{t}`' for t in type_val])}")
                else:
                    output.append(f"**Type**: `{type_val}`")
            
            # List other keys and their types
            output.append("\n## Properties")
            for key, value in current.items():
                if key in ['@id', '@type']:
                    continue
                    
                if isinstance(value, dict):
                    if '@id' in value:
                        output.append(f"- **{key}**: Reference to `{value['@id']}`")
                    else:
                        output.append(f"- **{key}**: Complex object with {len(value)} properties")
                elif isinstance(value, list):
                    output.append(f"- **{key}**: List with {len(value)} items")
                else:
                    output.append(f"- **{key}**: {value}")
            
            # Add a JSON representation
            json_str = json.dumps(current, indent=2)
            if len(json_str) > max_size:
                json_str = json_str[:max_size] + "..."
            output.append("\n## Raw JSON")
            output.append("```json")
            output.append(json_str)
            output.append("```")
            
            return "\n".join(output)
            
        elif isinstance(current, list):
            # For lists, show a summary and the first few items
            output = [f"# List at path: {path or 'root'}"]
            output.append(f"Contains {len(current)} items")
            
            # Show details for the first few items
            for i, item in enumerate(current[:5]):
                output.append(f"\n## Item {i+1}")
                if isinstance(item, dict):
                    if '@id' in item:
                        output.append(f"**ID**: `{item['@id']}`")
                    if '@type' in item:
                        type_val = item['@type']
                        if isinstance(type_val, list):
                            output.append(f"**Type**: {', '.join([f'`{t}`' for t in type_val])}")
                        else:
                            output.append(f"**Type**: `{type_val}`")
                    
                    # List a few key properties
                    keys = [k for k in item.keys() if k not in ['@id', '@type']]
                    if keys:
                        output.append("\n**Properties**:")
                        for key in keys[:5]:
                            value = item[key]
                            if isinstance(value, (dict, list)):
                                output.append(f"- **{key}**: Complex value")
                            else:
                                output.append(f"- **{key}**: {value}")
                        
                        if len(keys) > 5:
                            output.append(f"- ... and {len(keys) - 5} more properties")
                else:
                    output.append(f"Value: {item}")
            
            if len(current) > 5:
                output.append(f"\n... and {len(current) - 5} more items")
            
            # Add a JSON representation of the first few items
            json_str = json.dumps(current[:5], indent=2)
            if len(json_str) > max_size:
                json_str = json_str[:max_size] + "..."
            output.append("\n## Raw JSON (first 5 items)")
            output.append("```json")
            output.append(json_str)
            output.append("```")
            
            return "\n".join(output)
            
        else:
            # For simple values, just show the value
            return f"# Value at path: {path or 'root'}\n\n{current}"
            
    except (KeyError, IndexError, TypeError) as e:
        return f"Error accessing path '{path}': {str(e)}"

In [ ]:
#| test
def test_explore_dataset():
    # Setup a test dataset
    global dataset_kb
    dataset_kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "recordSet": [
            {
                "@id": "dataset/recordset1",
                "@type": "cr:RecordSet",
                "name": "Test RecordSet",
                "field": [
                    {
                        "@id": "dataset/field1",
                        "@type": "cr:Field",
                        "name": "field1",
                        "dataType": ["schema:Text"]
                    },
                    {
                        "@id": "dataset/field2",
                        "@type": "cr:Field",
                        "name": "field2",
                        "dataType": ["schema:Number"]
                    }
                ]
            }
        ],
        "distribution": [
            {
                "@id": "dataset/dist1",
                "@type": "schema:DataDownload",
                "contentUrl": "https://example.org/data.csv",
                "encodingFormat": "text/csv"
            }
        ]
    })
    
    # Test 1: Root level exploration
    result = explore_dataset()
    assert("Structure at path: root" in result)
    assert("recordSet" in result)
    assert("distribution" in result)
    
    # Test 2: Explore a specific path
    result = explore_dataset("recordSet[0]")
    assert("Test RecordSet" in result)
    assert("field" in result)
    
    # Test 3: Explore a list
    result = explore_dataset("recordSet[0].field")
    assert("Contains 2 items" in result)
    assert("field1" in result)
    assert("field2" in result)
    
    # Test 4: Explore a specific item in a list
    result = explore_dataset("recordSet[0].field[0]")
    assert("dataset/field1" in result)
    assert("dataType" in result)
    
    # Test 5: Explore a simple value
    result = explore_dataset("recordSet[0].name")
    assert("Test RecordSet" in result)
    
    # Test 6: Invalid path
    result = explore_dataset("invalid.path")
    assert("Error accessing path" in result)
    
    # Test 7: No dataset loaded
    # Save a reference to the dataset before deleting it
    saved_dataset = dataset_kb
    del globals()['dataset_kb']
    result = explore_dataset()
    assert("No dataset loaded" in result)
    
    # Restore dataset for other tests
    globals()['dataset_kb'] = saved_dataset



In [ ]:
test_explore_dataset()

In [ ]:
#| export
@tool
def search_dataset(
    query: str,  # Text to search for in the dataset
    case_sensitive: bool = False  # Whether search should be case sensitive
) -> str:  # Search results
    """Search for text in the dataset and return matching paths.
    
    Args:
        query: Text to search for in the dataset
        case_sensitive: Whether search should be case sensitive
        
    Returns:
        A markdown-formatted list of paths where the query was found
    """
    # Get the dataset knowledge base from the global context
    if 'dataset_kb' not in globals():
        return "No dataset loaded. Please load a dataset first."
    
    dataset_kb = globals()['dataset_kb']
    
    # Search results will be stored as (path, context)
    results = []
    
    def search_obj(obj, path=""):
        """Recursively search through an object"""
        if isinstance(obj, dict):
            # Search in keys
            for k, v in obj.items():
                # Check if key contains the query
                k_str = str(k)
                if (query in k_str) if case_sensitive else (query.lower() in k_str.lower()):
                    results.append((f"{path}.{k}" if path else k, "key"))
                
                # Search in values
                search_obj(v, f"{path}.{k}" if path else k)
                
        elif isinstance(obj, list):
            # Search in list items
            for i, item in enumerate(obj):
                search_obj(item, f"{path}[{i}]")
                
        elif isinstance(obj, str):
            # Search in string values
            if (query in obj) if case_sensitive else (query.lower() in obj.lower()):
                # Truncate long strings for display
                display_val = obj[:50] + "..." if len(obj) > 50 else obj
                results.append((path, display_val))
    
    # Start the search from the root
    search_obj(dataset_kb.data)
    
    if not results:
        return f"No matches found for '{query}'"
    
    # Format the results
    output = [f"# Found {len(results)} matches for '{query}'"]
    
    # Group results by type
    key_matches = [r for r in results if r[1] == "key"]
    value_matches = [r for r in results if r[1] != "key"]
    
    if key_matches:
        output.append("\n## Matching keys")
        for path, _ in key_matches[:10]:
            output.append(f"- `{path}`")
        if len(key_matches) > 10:
            output.append(f"- ... and {len(key_matches) - 10} more")
    
    if value_matches:
        output.append("\n## Matching values")
        for path, value in value_matches[:10]:
            output.append(f"- `{path}` = \"{value}\"")
        if len(value_matches) > 10:
            output.append(f"- ... and {len(value_matches) - 10} more")
    
    return "\n".join(output)

In [ ]:
#| test
def test_search_dataset():
    # Setup a test dataset
    global dataset_kb
    dataset_kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "recordSet": [
            {
                "@id": "dataset/recordset1",
                "@type": "cr:RecordSet",
                "name": "Test RecordSet",
                "description": "This is a test dataset for searching",
                "field": [
                    {
                        "@id": "dataset/field1",
                        "@type": "cr:Field",
                        "name": "field1",
                        "description": "A text field for testing",
                        "dataType": ["schema:Text"]
                    },
                    {
                        "@id": "dataset/field2",
                        "@type": "cr:Field",
                        "name": "field2",
                        "description": "A numeric field for testing",
                        "dataType": ["schema:Number"]
                    }
                ]
            }
        ]
    })
    
    # Test 1: Search for a term that appears in multiple places
    result = search_dataset("test")
    assert("Found" in result)
    assert("Test RecordSet" in result)
    assert("testing" in result)
    
    # Test 2: Case sensitive search
    result = search_dataset("Test", case_sensitive=True)
    assert("Found" in result)
    assert("Test RecordSet" in result)
    # Should not find "testing" with case_sensitive=True
    assert("testing" not in result)
    
    # Test 3: Search for a term in keys
    result = search_dataset("name")
    assert("Matching keys" in result)
    
    # Test 4: Search for a non-existent term
    result = search_dataset("nonexistent")
    assert("No matches found" in result)
    
    # Test 5: No dataset loaded
    saved_dataset = dataset_kb
    del globals()['dataset_kb']
    result = search_dataset("test")
    assert("No dataset loaded" in result)
    
    # Restore dataset for other tests
    globals()['dataset_kb'] = saved_dataset


In [ ]:
test_search_dataset()

In [ ]:
#| export
@tool
def get_dataset_evidence(
    topic: str,  # Topic to find evidence about
    max_results: int = 5  # Maximum number of evidence items to return
) -> str:  # Evidence from the dataset
    """Find evidence in the dataset about a specific topic.
    
    Args:
        topic: Topic to find evidence about
        max_results: Maximum number of evidence items to return
        
    Returns:
        A markdown-formatted collection of evidence about the topic
    """
    # Get the dataset knowledge base from the global context
    if 'dataset_kb' not in globals():
        return "No dataset loaded. Please load a dataset first."
    
    dataset_kb = globals()['dataset_kb']
    
    # First use search_dataset to find relevant paths
    search_results = search_dataset(topic)
    
    if "No matches found" in search_results:
        return f"No evidence found for topic: '{topic}'"
    
    # Extract paths from search results
    import re
    path_pattern = r'`([^`]+)`'
    paths = re.findall(path_pattern, search_results)
    
    # Collect evidence
    evidence = []
    
    for path in paths:
        # Skip paths that are just key names
        if path.count('.') == 0 and '[' not in path:
            continue
            
        try:
            # Navigate to the path
            current = dataset_kb.data
            parts = path.split('.')
            
            for part in parts:
                if '[' in part and part.endswith(']'):
                    name, idx_str = part.split('[', 1)
                    idx = int(idx_str[:-1])
                    if name:
                        current = current[name][idx]
                    else:
                        current = current[idx]
                else:
                    current = current[part]
            
            # Add context to the evidence
            if isinstance(current, dict):
                # For dictionaries, include ID and type if available
                context = {}
                if '@id' in current:
                    context['id'] = current['@id']
                if '@type' in current:
                    context['type'] = current['@type']
                if 'name' in current:
                    context['name'] = current['name']
                
                evidence.append({
                    'path': path,
                    'context': context,
                    'value': current
                })
            else:
                # For simple values, include the parent object for context
                parent_path = '.'.join(path.split('.')[:-1])
                last_part = path.split('.')[-1]
                
                parent = dataset_kb.data
                for part in parent_path.split('.'):
                    if '[' in part and part.endswith(']'):
                        name, idx_str = part.split('[', 1)
                        idx = int(idx_str[:-1])
                        if name:
                            parent = parent[name][idx]
                        else:
                            parent = parent[idx]
                    else:
                        parent = parent[part]
                
                context = {'property': last_part}
                if isinstance(parent, dict):
                    if '@id' in parent:
                        context['id'] = parent['@id']
                    if '@type' in parent:
                        context['type'] = parent['@type']
                    if 'name' in parent:
                        context['name'] = parent['name']
                
                evidence.append({
                    'path': path,
                    'context': context,
                    'value': current
                })
        except (KeyError, IndexError, TypeError) as e:
            # Skip paths that can't be resolved
            continue
    
    # Limit the number of evidence items
    evidence = evidence[:max_results]
    
    if not evidence:
        return f"No structured evidence found for topic: '{topic}'"
    
    # Format the evidence
    output = [f"# Evidence for topic: '{topic}'"]
    
    for i, item in enumerate(evidence):
        output.append(f"\n## Evidence {i+1}")
        output.append(f"**Path**: `{item['path']}`")
        
        # Add context information
        if item['context']:
            output.append("\n**Context**:")
            for k, v in item['context'].items():
                output.append(f"- {k}: {v}")
        
        # Add the value
        output.append("\n**Value**:")
        if isinstance(item['value'], dict):
            # For dictionaries, show a summary
            keys = list(item['value'].keys())
            output.append(f"Object with {len(keys)} properties:")
            for k in keys[:5]:
                v = item['value'][k]
                if isinstance(v, (dict, list)):
                    output.append(f"- {k}: (complex value)")
                else:
                    output.append(f"- {k}: {v}")
            if len(keys) > 5:
                output.append(f"- ... and {len(keys) - 5} more properties")
        elif isinstance(item['value'], list):
            # For lists, show a summary
            output.append(f"List with {len(item['value'])} items")
        else:
            # For simple values, show the value
            output.append(str(item['value']))
    
    return "\n".join(output)

In [ ]:
#| test
def test_get_dataset_evidence():
    # Setup a test dataset
    global dataset_kb
    dataset_kb = LinkedDataKnowledge({
        "@context": {"schema": "https://schema.org/"},
        "recordSet": [
            {
                "@id": "dataset/recordset1",
                "@type": "cr:RecordSet",
                "name": "Test RecordSet",
                "description": "This is a test dataset about machine learning",
                "field": [
                    {
                        "@id": "dataset/field1",
                        "@type": "cr:Field",
                        "name": "feature1",
                        "description": "A machine learning feature",
                        "dataType": ["schema:Text"]
                    },
                    {
                        "@id": "dataset/field2",
                        "@type": "cr:Field",
                        "name": "feature2",
                        "description": "Another machine learning feature",
                        "dataType": ["schema:Number"]
                    }
                ]
            }
        ],
        "keywords": ["machine learning", "dataset", "test"]
    })
    
    # Test 1: Get evidence about a topic
    result = get_dataset_evidence("machine learning")
    assert("Evidence for topic: 'machine learning'" in result)
    assert("This is a test dataset about machine learning" in result)
    
    # Test 2: Get evidence with max_results
    result = get_dataset_evidence("feature", max_results=1)
    assert("Evidence for topic: 'feature'" in result)
    assert("Evidence 1" in result)
    assert("Evidence 2" not in result)
    
    # Test 3: No evidence found
    result = get_dataset_evidence("nonexistent")
    assert("No evidence found" in result)
    
    # Test 4: No dataset loaded
    saved_dataset = dataset_kb
    del globals()['dataset_kb']
    result = get_dataset_evidence("machine learning")
    assert("No dataset loaded" in result)
    
    # Restore dataset for other tests
    globals()['dataset_kb'] = saved_dataset


In [ ]:
test_get_dataset_evidence()

In [ ]:
@tool
def collect_evidence(topic:str, observation:str, source_term:str=None, importance:str="medium") -> str:
    "Collect evidence about a topic during linked data exploration"
    global kb
    
    if not hasattr(kb, 'evidence_collection'): kb.evidence_collection = []
    
    evidence_item = {
        "topic": topic,
        "observation": observation,
        "source": source_term,
        "importance": importance,
        "timestamp": datetime.datetime.now().isoformat()
    }
    
    kb.evidence_collection.append(evidence_item)
    
    return f"Evidence collected about '{topic}' (importance: {importance}). Total evidence items: {len(kb.evidence_collection)}"


In [ ]:
@tool
def summarize_evidence(topic:str=None) -> str:
    "Summarize collected evidence, optionally filtered by topic"
    global kb
    
    if not hasattr(kb, 'evidence_collection') or not kb.evidence_collection:
        return "No evidence has been collected yet."
    
    # Filter by topic if provided
    if topic:
        evidence_items = [item for item in kb.evidence_collection if topic.lower() in item["topic"].lower()]
    else:
        evidence_items = kb.evidence_collection
    
    if not evidence_items:
        return f"No evidence found for topic '{topic}'."
    
    # Group evidence by topic
    topics = {}
    for item in evidence_items:
        if item["topic"] not in topics: topics[item["topic"]] = []
        topics[item["topic"]].append(item)
    
    # Format the evidence as markdown
    result = f"# Evidence Collection Summary ({len(evidence_items)} items)\n\n"
    
    # First, list high importance items
    high_importance = [item for item in evidence_items if item["importance"] == "high"]
    if high_importance:
        result += "## Key Findings\n"
        for item in high_importance:
            result += f"- **{item['topic']}**: {item['observation']}\n"
        result += "\n"
    
    # Then organize by topic
    result += "## Detailed Findings by Topic\n\n"
    for topic, items in topics.items():
        result += f"### {topic} ({len(items)} findings)\n"
        for item in items:
            result += f"- {item['observation']}"
            if item['source']: result += f" (Source: {item['source']})"
            result += "\n"
        result += "\n"
    
    # Add a section for relationships between topics
    result += "## Relationships Between Topics\n"
    result += "Consider how these topics relate to each other to form a complete data model.\n\n"
    
    return result

In [ ]:
# Test evidence collection and summarization
import datetime
global kb
kb = LinkedDataKnowledge()
kb.evidence_collection = []

# Collect some evidence
result1 = collect_evidence("Person", "A Person represents an individual human being", "schema:Person", "high")
print(result1)
result2 = collect_evidence("Organization", "Organizations can employ Persons", "schema:Organization")
print(result2)
result3 = collect_evidence("Person", "A Person can have credentials", "schema:hasCredential")
print(result3)

# Test evidence count
test_eq(len(kb.evidence_collection), 3)

# Test summarization
summary = summarize_evidence()
print(summary)
test_eq("Person" in summary, True)
test_eq("Organization" in summary, True)
test_eq("Key Findings" in summary, True)  # Should include high importance items

# Test topic filtering
person_summary = summarize_evidence("Person")
print(person_summary)
test_eq(len(person_summary.split("\n")) > 5, True)  # Should have multiple lines
test_eq("Organization" in person_summary, False)  # Should not include other topics

Evidence collected about 'Person' (importance: high). Total evidence items: 1
Evidence collected about 'Organization' (importance: medium). Total evidence items: 2
Evidence collected about 'Person' (importance: medium). Total evidence items: 3
# Evidence Collection Summary (3 items)

## Key Findings
- **Person**: A Person represents an individual human being

## Detailed Findings by Topic

### Person (2 findings)
- A Person represents an individual human being (Source: schema:Person)
- A Person can have credentials (Source: schema:hasCredential)

### Organization (1 findings)
- Organizations can employ Persons (Source: schema:Organization)

## Relationships Between Topics
Consider how these topics relate to each other to form a complete data model.


# Evidence Collection Summary (2 items)

## Key Findings
- **Person**: A Person represents an individual human being

## Detailed Findings by Topic

### Person (2 findings)
- A Person represents an individual human being (Source: schema:Pe

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()